# Feature Engineering

## Table of Contents
#### 1. Importing libraries
#### 2. Categorizing products into price range groups for easy filtering
##### 2.1. If-Statements with User-Defined Functions
##### 2.2. If-Statements with the loc Function (preferable way)
#### 3. Creating the 'busiest_day' column
##### 3.1. If-Statements with For-Loops
#### 4. Creating the 'busiest_days' column
#### 5. Creating the column "busiest_period_of_day"

## 1. Importing libraries

In [10]:
# Import libraries

import pandas as pd
import numpy as np
import os

In [11]:
# Main path
path = r'/Users/helenarobalinho/Desktop/Data Analytics/2. Data Immersion/Achievement 4 - Python Fundamentals for Data Analysts/03-2026 Instacart Basket Analysis'

## 2. Categorizing products into price range groups for easy filtering

### 2.1. If-Statements with User-Defined Functions

#### Categorize products into price range groups for easy filtering

In [15]:
# Import the orders_products_merge df from the pickle file
ords_prods_merge = pd.read_pickle(os.path.join(path, '02 Data', 'Prepared Data', 'ords_prods_merge.pkl'))

In [16]:
# Subset to avoid memory issues
df = ords_prods_merge [:1000000]
# the subset should include the first one million rows in the dataframe

In [17]:
# Define a function
def price_label (row):
    
    if row['prices'] <= 5:
        return 'Low-range product'
    elif (row['prices'] > 5) and (row['prices'] <= 15):
        return 'Mid-range product'
    elif row['prices'] > 15:
        return 'High range'
    else: return 'Not enough data'

In [18]:
# defining the function by using the def syntax at the beginning of the code. Following is the name you want to give your new function: price_label. 
# In the parentheses is row, which is a standard argument telling the function to look at each row within the dataframe. 

# If the value within the "prices" column within the given row is less than or equal to 5, then
    # return the string "Low-range product."
# Or else, if the value within the "prices" column within the given row is greater than 5 and less than or equal to 15, 
    # then return the string "Mid-range product."
# Or else, if the value of the "prices" column within the given row is greater than 15, then
    # return the string "High-range product."
# Or else, return the string "Not enough data."

In [19]:
# Apply the function
df ['price_range'] = df.apply(price_label, axis=1)

/var/folders/ff/tlfsdsyn5jnfl92hv4dxd79m0000gn/T/ipykernel_2929/3277225944.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df ['price_range'] = df.apply(price_label, axis=1)


In [20]:
# Check the values in the new column
df ['price_range'].value_counts (dropna = False)

price_range
Mid-range product    652638
Low-range product    338018
High range             9344
Name: count, dtype: int64

In [21]:
# Check what the most expensice product within the subset
df ['prices'].max()

24.5

### 2.2. If-Statements with the loc Function (preferable way)

In [22]:
# Create the conditions:

In [23]:
df.loc[df['prices'] > 15, 'price_range_loc'] = 'High-range product'

/var/folders/ff/tlfsdsyn5jnfl92hv4dxd79m0000gn/T/ipykernel_2929/1169838859.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.loc[df['prices'] > 15, 'price_range_loc'] = 'High-range product'


In [24]:
df.loc[(df['prices'] <= 15) & (df['prices'] > 5), 'price_range_loc'] = 'Mid-range product'

In [25]:
df.loc[df['prices'] <= 5, 'price_range_loc'] = 'Low-range product'

In [26]:
df['price_range_loc'].value_counts(dropna = False)

price_range_loc
Mid-range product     652638
Low-range product     338018
High-range product      9344
Name: count, dtype: int64

In [27]:
# if = df.loc[df['prices'] > 15,
    # then = 'price_range_loc'] = "High-range product'

- The loc() method runs much faster; 
- The loc() function applies the conditional filters before searching through the datatrame, while your user-defined function searches through the entire dataframe and then determines where to set the filters (remember axis = 1?).

### Repeating the process for the entire dataframe (by using user-defined function, there would be a memory error!)

In [28]:
# Create the conditions:

In [29]:
ords_prods_merge.loc[ords_prods_merge['prices'] > 15, 'price_range_loc'] = 'High-range product'

In [30]:
ords_prods_merge.loc[(ords_prods_merge['prices'] <= 15) & (ords_prods_merge['prices'] > 5), 'price_range_loc'] = 'Mid-range product'

In [31]:
ords_prods_merge.loc[ords_prods_merge['prices'] <= 5, 'price_range_loc'] = 'Low-range product'

In [32]:
ords_prods_merge['price_range_loc'].value_counts(dropna = False)

price_range_loc
Mid-range product     21860860
Low-range product     10126321
High-range product      417678
Name: count, dtype: int64

In [33]:
# The sum of the above values is 32 404 859

## 3. Creating the 'busiest_day' column

Create a new column in your ords_prods_merge dataframe that summarizes how busy each day of the week is. This information is valuable information for stakeholders as it gives them insignt into what products are being bougnt on the busiest and slowest days. They could use this information to tailor ads on specific days.

#### 3.1. If-Statements with For-Loops

In [37]:
# find which day most orders take place
# print the frequency of the 'orders_day_of_week' column

ords_prods_merge['orders_day_of_week'].value_counts(dropna = False)

orders_day_of_week
0    6204182
1    5660230
6    4496490
2    4213830
5    4205791
3    3840534
4    3783802
Name: count, dtype: int64

In [38]:
# 0 means Saturday, it is the busiest day
# 4 means Wednesday, it is the slowest day for Instacart app orders

In [39]:
# create a new column, "busiest day," that will contain one of three different values:
# "Busiest day," "Least busy," and "Regularly busy." 

result = []
for value in ords_prods_merge["orders_day_of_week"]:
    if value == 0:
        result.append("Busiest day")
    elif value == 4:
        result. append ("Least busy")
    else:
        result. append ("Regularly busy")

In [40]:
# Print the result
result

['Regularly busy',
 'Regularly busy',
 'Busiest day',
 'Regularly busy',
 'Least busy',
 'Regularly busy',
 'Regularly busy',
 'Regularly busy',
 'Regularly busy',
 'Regularly busy',
 'Regularly busy',
 'Regularly busy',
 'Least busy',
 'Regularly busy',
 'Regularly busy',
 'Regularly busy',
 'Regularly busy',
 'Regularly busy',
 'Regularly busy',
 'Regularly busy',
 'Regularly busy',
 'Regularly busy',
 'Regularly busy',
 'Regularly busy',
 'Regularly busy',
 'Busiest day',
 'Busiest day',
 'Busiest day',
 'Busiest day',
 'Busiest day',
 'Least busy',
 'Regularly busy',
 'Regularly busy',
 'Busiest day',
 'Regularly busy',
 'Regularly busy',
 'Least busy',
 'Regularly busy',
 'Busiest day',
 'Regularly busy',
 'Busiest day',
 'Least busy',
 'Busiest day',
 'Regularly busy',
 'Regularly busy',
 'Regularly busy',
 'Regularly busy',
 'Regularly busy',
 'Regularly busy',
 'Regularly busy',
 'Regularly busy',
 'Regularly busy',
 'Regularly busy',
 'Regularly busy',
 'Regularly busy',
 'Bus

In [41]:
# create a new column within your ords_prods_merge dataframe 
# and set it equal to result

ords_prods_merge['busiest_day'] = result

In [42]:
# frequency
ords_prods_merge ['busiest_day'].value_counts(dropna = False)

busiest_day
Regularly busy    22416875
Busiest day        6204182
Least busy         3783802
Name: count, dtype: int64

In [43]:
# The sum of the above values is 32 404 859

## 4. Creating the 'busiest_days' column

Suppose your clients have changed their minds about the labels you created in your "busiest_day" column. Now, they want "Busiest day" to become "Busiest days" (plural). This label should correspond with the two busiest days of the week as opposed to the single busiest day. At the same time, they'd also like to know the two slowest days. Create a new column for this using a suitable method.

In [44]:
# I already have the info about the two busiest days: Saturday (0) and Sunday (1)
# Whereas Wednesday (3) and Thursday (4) are the slowest days

# So, what I need is to add a logical operator (or) to the for loop we used a few lines above

# On every loop, the line 'if value in (0, 1)' will check if the day is either Saturday (0) OR Sunday (1); 
# If it is, then it will add 'Busiest days' to the 'results = []' list.

# If it's not Saturday OR Sunday, the second line (elif value in(3, 4:) will execute and check whether the day 
# is either Wednesday (3) or Thursday (4). 
# If so, 'Slowest days' will be pushed to the 'results = []' list.

# If none of the two previous conditions is met, then the the 'else' will execute, meaning that 'Regular days' will be pushed to the results list.

result_2 = []
for value in ords_prods_merge["orders_day_of_week"]:
    if value == 0 or value == 1:
        result_2.append("Busiest days")
    elif value == 3 or value == 4:
        result_2. append ("Slowest days")
    else:
        result_2. append ("Regular days")

In [45]:
result_2

['Regular days',
 'Regular days',
 'Busiest days',
 'Slowest days',
 'Slowest days',
 'Busiest days',
 'Regular days',
 'Slowest days',
 'Busiest days',
 'Busiest days',
 'Regular days',
 'Slowest days',
 'Slowest days',
 'Regular days',
 'Slowest days',
 'Regular days',
 'Regular days',
 'Regular days',
 'Busiest days',
 'Busiest days',
 'Regular days',
 'Busiest days',
 'Busiest days',
 'Busiest days',
 'Busiest days',
 'Busiest days',
 'Busiest days',
 'Busiest days',
 'Busiest days',
 'Busiest days',
 'Slowest days',
 'Regular days',
 'Busiest days',
 'Busiest days',
 'Regular days',
 'Regular days',
 'Slowest days',
 'Busiest days',
 'Busiest days',
 'Busiest days',
 'Busiest days',
 'Slowest days',
 'Busiest days',
 'Busiest days',
 'Busiest days',
 'Slowest days',
 'Regular days',
 'Regular days',
 'Busiest days',
 'Busiest days',
 'Busiest days',
 'Slowest days',
 'Regular days',
 'Busiest days',
 'Regular days',
 'Busiest days',
 'Busiest days',
 'Regular days',
 'Busiest days

In [46]:
# Creating the 'busiest_days' column within the ords_prods_merge dataframe 

ords_prods_merge['busiest_days'] = result_2

#### Checking the values of this new column for accuracy

In [47]:
ords_prods_merge['busiest_days'].value_counts(dropna = False)

busiest_days
Regular days    12916111
Busiest days    11864412
Slowest days     7624336
Name: count, dtype: int64

In [48]:
# The sum of the above values is 32 404 859

In [49]:
# Check output of ords_prods_merge with new "busiest days" column
ords_prods_merge.head()

,product_id,product_name,aisle_id,department_id,prices,Unnamed: 0.1,Unnamed: 0,order_id,user_id,eval_set,...,orders_day_of_week,order_hour_of_day,days_since_prior_order,First_order,add_to_cart_order,reordered,_merge,price_range_loc,busiest_day,busiest_days
0,1,Chocolate Sandwich Cookies,61,19,5.8,1987,1987,3139998,138,prior,...,6,11,3.0,False,5,0,both,Mid-range product,Regularly busy,Regular days
1,1,Chocolate Sandwich Cookies,61,19,5.8,1989,1989,1977647,138,prior,...,6,17,20.0,False,1,1,both,Mid-range product,Regularly busy,Regular days
2,1,Chocolate Sandwich Cookies,61,19,5.8,11433,11433,389851,709,prior,...,0,21,6.0,False,20,0,both,Mid-range product,Busiest day,Busiest days
3,1,Chocolate Sandwich Cookies,61,19,5.8,12198,12198,652770,764,prior,...,3,13,NaN,True,10,0,both,Mid-range product,Regularly busy,Slowest days
4,1,Chocolate Sandwich Cookies,61,19,5.8,12200,12200,1813452,764,prior,...,4,17,9.0,False,11,1,both,Mid-range product,Least busy,Slowest days


- Observations: The total value counts for the ords_prods_merge dataframe is equal to the total sum of all groupings (i.e. "Regularly busy" + "Busiest days" + "Slowest days"). In addition, the listed sums for each label match the sum totals for the qualifying days of the week (e.g. Total "Busiest days" = 11864412, which matches orders_day_of_week 0 + 1, or 6204182 + 5660230).

## 5. Creating the column "busiest_period_of_day"

When too many users make Instacart orders at the same time, the app freezes. The senior technical officer at Instacart wants you to identify the busiest hours of the day. Rather than by hour, they want periods of time labeled "Most orders," "Average orders," and "Fewest orders." Create a new column containing these labels called "busiest_period_of_day."

In [50]:
# First, let's check the value counts in "order_hour_of_day" column.
ords_prods_merge['order_hour_of_day'].value_counts(dropna = False)

order_hour_of_day
10    2761760
11    2736140
14    2689136
15    2662144
13    2660954
12    2618532
16    2535202
9     2454203
17    2087654
8     1718118
18    1636502
19    1258305
20     976156
7      891054
21     795637
22     634225
23     402316
6      290493
0      218769
1      115700
5       87961
2       69375
4       53242
3       51281
Name: count, dtype: int64

In [51]:
# We now have a list from the busiest hour to the least busy one

#### The value counts listed above will be split into equal thirds for the following labels: 
#### "Most orders" = 10, 11, 14, 15, 13, 12, 16, and 9; 
#### "Fewest orders" = 23, 6, 0, 1, 5, 2, 4, and 3; 
#### "Average orders" = all remaining values in order_hour_of_day.

In [52]:
# Create a for-loop if-statement labeling the periods of time:

results_hours = []

for value in ords_prods_merge["order_hour_of_day"]:
    if value in (9, 10, 11, 12, 13, 14, 15, 16):
        results_hours.append("Most orders")
    elif value in (7, 8, 17, 18, 19, 20, 21, 22):
        results_hours.append("Average orders")
    else:
        results_hours.append("Fewest orders")

In [53]:
results_hours

['Most orders',
 'Average orders',
 'Average orders',
 'Most orders',
 'Average orders',
 'Average orders',
 'Most orders',
 'Most orders',
 'Average orders',
 'Most orders',
 'Most orders',
 'Most orders',
 'Most orders',
 'Most orders',
 'Most orders',
 'Average orders',
 'Average orders',
 'Fewest orders',
 'Average orders',
 'Fewest orders',
 'Fewest orders',
 'Fewest orders',
 'Fewest orders',
 'Most orders',
 'Most orders',
 'Average orders',
 'Average orders',
 'Average orders',
 'Average orders',
 'Average orders',
 'Average orders',
 'Average orders',
 'Fewest orders',
 'Average orders',
 'Most orders',
 'Most orders',
 'Average orders',
 'Most orders',
 'Most orders',
 'Most orders',
 'Average orders',
 'Most orders',
 'Most orders',
 'Most orders',
 'Average orders',
 'Most orders',
 'Average orders',
 'Most orders',
 'Most orders',
 'Most orders',
 'Most orders',
 'Most orders',
 'Most orders',
 'Most orders',
 'Average orders',
 'Average orders',
 'Most orders',
 'Most ord

In [54]:
# Creating the new column "busiest_period_of_day" in ords_prods_merge df

ords_prods_merge['busiest_period_of_day'] = results_hours

#### Printing the frequency for this new column

In [55]:
# Print value counts in "busiest_period_of_day" column

ords_prods_merge['busiest_period_of_day'].value_counts(dropna = False)

busiest_period_of_day
Most orders       21118071
Average orders     9997651
Fewest orders      1289137
Name: count, dtype: int64

In [56]:
# The sum of the above values is also equal to = 32 404 859

In [57]:
ords_prods_merge.head()

,product_id,product_name,aisle_id,department_id,prices,Unnamed: 0.1,Unnamed: 0,order_id,user_id,eval_set,...,order_hour_of_day,days_since_prior_order,First_order,add_to_cart_order,reordered,_merge,price_range_loc,busiest_day,busiest_days,busiest_period_of_day
0,1,Chocolate Sandwich Cookies,61,19,5.8,1987,1987,3139998,138,prior,...,11,3.0,False,5,0,both,Mid-range product,Regularly busy,Regular days,Most orders
1,1,Chocolate Sandwich Cookies,61,19,5.8,1989,1989,1977647,138,prior,...,17,20.0,False,1,1,both,Mid-range product,Regularly busy,Regular days,Average orders
2,1,Chocolate Sandwich Cookies,61,19,5.8,11433,11433,389851,709,prior,...,21,6.0,False,20,0,both,Mid-range product,Busiest day,Busiest days,Average orders
3,1,Chocolate Sandwich Cookies,61,19,5.8,12198,12198,652770,764,prior,...,13,NaN,True,10,0,both,Mid-range product,Regularly busy,Slowest days,Most orders
4,1,Chocolate Sandwich Cookies,61,19,5.8,12200,12200,1813452,764,prior,...,17,9.0,False,11,1,both,Mid-range product,Least busy,Slowest days,Average orders


In [58]:
ords_prods_merge.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32404859 entries, 0 to 32404858
Data columns (total 22 columns):
 #   Column                  Dtype   
---  ------                  -----   
 0   product_id              int64   
 1   product_name            object  
 2   aisle_id                int64   
 3   department_id           int64   
 4   prices                  float64 
 5   Unnamed: 0.1            int64   
 6   Unnamed: 0              int64   
 7   order_id                int64   
 8   user_id                 int64   
 9   eval_set                object  
 10  order_number            int64   
 11  orders_day_of_week      int64   
 12  order_hour_of_day       int64   
 13  days_since_prior_order  float64 
 14  First_order             bool    
 15  add_to_cart_order       int64   
 16  reordered               int64   
 17  _merge                  category
 18  price_range_loc         object  
 19  busiest_day             object  
 20  busiest_days            object  
 21  busies

In [59]:
# Columns "eval_set", "Unnamed: 0.1" and "Unnamed: 0" can be cleaned

In [60]:
# Creating new df without the unnecessary columns:
ords_prods_merge = ords_prods_merge.drop(columns=['eval_set', 'Unnamed: 0', 'Unnamed: 0.1'])

In [61]:
ords_prods_merge.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32404859 entries, 0 to 32404858
Data columns (total 19 columns):
 #   Column                  Dtype   
---  ------                  -----   
 0   product_id              int64   
 1   product_name            object  
 2   aisle_id                int64   
 3   department_id           int64   
 4   prices                  float64 
 5   order_id                int64   
 6   user_id                 int64   
 7   order_number            int64   
 8   orders_day_of_week      int64   
 9   order_hour_of_day       int64   
 10  days_since_prior_order  float64 
 11  First_order             bool    
 12  add_to_cart_order       int64   
 13  reordered               int64   
 14  _merge                  category
 15  price_range_loc         object  
 16  busiest_day             object  
 17  busiest_days            object  
 18  busiest_period_of_day   object  
dtypes: bool(1), category(1), float64(2), int64(10), object(5)
memory usage: 4.2+ GB


### Exporting derived dataframe

In [62]:
# Export dataframe as a pickle file to “Prepared Data” folder:

ords_prods_merge.to_pickle(os.path.join(path, '02 Data', 'Prepared Data', 'orders_products_merged_derived.pkl'))